In [1]:
from langchain_community.retrievers import BM25Retriever
from datasets import Dataset
import numpy as np
import json

/u/modelfactory/.conda/envs/lab/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
base_data_path = 'data/industrial/processed'

In [3]:
with open(f'{base_data_path}/all_tasks_dataset.json', 'r') as f:
    data = json.load(f)

In [4]:
id_to_corpus = dict(zip(data['corpus_to_id'].values(), data['corpus_to_id'].keys()))

In [5]:
train_dataset = Dataset.from_json(f'{base_data_path}/train.json')
val_dataset = Dataset.from_json(f'{base_data_path}/val.json')
test_dataset = Dataset.from_json(f'{base_data_path}/test.json')

In [6]:
all_docs = np.unique([item['sentence2'] for dataset in [train_dataset, val_dataset, test_dataset] for item in dataset])

In [7]:
retriever = BM25Retriever.from_texts(all_docs.tolist(), k=100)

In [8]:
all_tasks = np.unique(test_dataset['task'])

In [55]:
# for question in all_test_questions:
#     qid = data['query_to_id'][question]
#     doc_ids = data['query_to_doc'][str(qid)]
#     gt_docs = [id_to_corpus[doc_id] for doc_id in doc_ids]
#     retrieved_docs = retriever.invoke(question)
#     # todo: calculate the metrics MAP, NDCG, Accuracy
#     break


In [9]:
def compute_dcg_at_k(relevances, k):
    dcg = 0
    for i in range(min(len(relevances), k)):
        dcg += relevances[i] / np.log2(i + 2)  # +2 as we start our idx at 0
    return dcg

In [10]:
def compute_metrics(all_test_questions):
    accuracy_at_k = precision_recall_at_k = [1, 3, 5, 10]
    mrr_at_k = [10]
    ndcg_at_k = [10]
    map_at_k = [100]
    num_hits_at_k = {k: 0 for k in accuracy_at_k}
    precisions_at_k = {k: [] for k in precision_recall_at_k}
    recall_at_k = {k: [] for k in precision_recall_at_k}
    MRR = {k: 0 for k in mrr_at_k}
    ndcg = {k: [] for k in ndcg_at_k}
    AveP_at_k = {k: [] for k in map_at_k}
    for query in all_test_questions:
        qid = data['query_to_id'][query]
        query_relevant_docs = data['query_to_doc'][str(qid)]
        gt_docs_str = [id_to_corpus[doc_id] for doc_id in query_relevant_docs]
        retrieved_docs = retriever.invoke(query)
        top_hits = [{'corpus_id': data['corpus_to_id'][retrieved_doc.page_content]} for retrieved_doc in retrieved_docs]
        # Sort scores
        # top_hits = sorted(queries_result_list[query_itr], key=lambda x: x["score"], reverse=True)
        # query_relevant_docs = relevant_docs[query_id]
        for k_val in accuracy_at_k:
            for hit in top_hits[0:k_val]:
                if hit["corpus_id"] in query_relevant_docs:
                    num_hits_at_k[k_val] += 1
                    break
        
        # Precision and Recall@k
        for k_val in precision_recall_at_k:
            num_correct = 0
            for hit in top_hits[0:k_val]:
                if hit["corpus_id"] in query_relevant_docs:
                    num_correct += 1
            
            precisions_at_k[k_val].append(num_correct / k_val)
            recall_at_k[k_val].append(num_correct / len(query_relevant_docs))
        
        # MRR@k
        for k_val in mrr_at_k:
            for rank, hit in enumerate(top_hits[0:k_val]):
                if hit["corpus_id"] in query_relevant_docs:
                    MRR[k_val] += 1.0 / (rank + 1)
                    break
        
        # NDCG@k
        for k_val in ndcg_at_k:
            predicted_relevance = [
                1 if top_hit["corpus_id"] in query_relevant_docs else 0 for top_hit in top_hits[0:k_val]
            ]
            true_relevances = [1] * len(query_relevant_docs)
        
            ndcg_value = compute_dcg_at_k(predicted_relevance, k_val) / compute_dcg_at_k(
                true_relevances, k_val
            )
            ndcg[k_val].append(ndcg_value)
        
        # MAP@k
        for k_val in map_at_k:
            num_correct = 0
            sum_precisions = 0
        
            for rank, hit in enumerate(top_hits[0:k_val]):
                if hit["corpus_id"] in query_relevant_docs:
                    num_correct += 1
                    sum_precisions += num_correct / (rank + 1)
        
            avg_precision = sum_precisions / min(k_val, len(query_relevant_docs))
            AveP_at_k[k_val].append(avg_precision)
        
    # Compute averages
    for k in num_hits_at_k:
        num_hits_at_k[k] /= len(all_test_questions)
    
    for k in precisions_at_k:
        precisions_at_k[k] = np.mean(precisions_at_k[k])
    
    for k in recall_at_k:
        recall_at_k[k] = np.mean(recall_at_k[k])
    
    for k in ndcg:
        ndcg[k] = np.mean(ndcg[k])
    
    for k in MRR:
        MRR[k] /= len(all_test_questions)
    
    for k in AveP_at_k:
        AveP_at_k[k] = np.mean(AveP_at_k[k])
    return {
        "accuracy@k": num_hits_at_k,
        "precision@k": precisions_at_k,
        "recall@k": recall_at_k,
        "ndcg@k": ndcg,
        "mrr@k": MRR,
        "map@k": AveP_at_k,
    }

In [12]:
all_scores = {}
for task in all_tasks:
    task_questions = np.unique([item['sentence1'] for item in test_dataset if item['label'] == 1 and item['task'] == task])
    all_scores[task] = compute_metrics(task_questions)

In [11]:
tasks = [
    'asset_to_related_sensors', 'component_to_failure_mode', 
    'eq_to_category', 'eq_to_class_type', 'eq_subunit_to_unit',
    'asset_fm_to_components', 'failure_desc_to_class', 'asset_fault_to_sensor', 'asset_sensor_to_fault'
]
metrics = [('accuracy@k', 1), ('map@k', 100), ('ndcg@k', 10)]

In [13]:
print('acc@1', 'map@100', 'ndcg@10')
for task in tasks:
    res = task
    for metric in metrics:
        res += ', ' + str(round(all_scores[task][metric[0]][metric[1]] * 100, 2))
    print(res)

acc@1 map@100 ndcg@10
asset_to_related_sensors, 0.0, 0.76, 2.42
component_to_failure_mode, 0.0, 0.3, 0.73
eq_to_category, 14.29, 4.61, 12.53
eq_to_class_type, 0.0, 1.08, 0.0
eq_subunit_to_unit, 0.0, 0.53, 0.55
asset_fm_to_components, 0.0, 0.0, 0.0
failure_desc_to_class, 8.14, 12.78, 14.55
asset_fault_to_sensor, 0.0, 2.84, 4.45
asset_sensor_to_fault, 0.0, 0.46, 0.0


In [50]:
with open('results/bm25.json', 'w') as f:
    f.write(json.dumps(all_scores))